# Задача 3. AVL

Реализовать AVL-дерево с операциями вставки, удаления и поиска.

Как в предыдущей работе с BST, дубликаты не добавляются:
слева значения меньше текущего узла, справа — больше.

## Объяснение решения

AVL — дерево поиска, которое поддерживает баланс после изменений.
У каждого узла высоты левого и правого поддеревьев отличаются не больше чем на 1.

Высота пустого дерева — 0, листа — 1.

`height = 1 + max(left_height, right_height)`

`balance = left_height - right_height`

При поиске идём влево или вправо по сравнению ключей.
При вставке сначала находим место нового листа, затем на пути назад
обновляем высоты. Если баланс стал 2 или −2, выполняем поворот.

При удалении рассматриваем три случая:

- нет детей — удаляем лист;
- один ребёнок — ставим его на место узла;
- два ребёнка — заменяем значение минимумом правого поддерева,
  затем удаляем прежний узел с этим минимумом.

После удаления также проверяем баланс всех предков.
Повороты сохраняют порядок ключей, поэтому поиск остаётся корректным.

## Класс узла

К узлу из предыдущей работы добавляем `height` — высоту его поддерева.

In [1]:
class Node:
    """Один узел дерева; height хранит высоту поддерева этого узла."""

    def __init__(self, value, left=None, right=None, height=1):
        self.value = value
        self.left = left
        self.right = right
        self.height = height

In [2]:
class AVL:
    """Дерево поиска с автоматическим поддержанием AVL-баланса.

    Структуру root и Node можно читать для изучения и визуализации.
    Менять её следует через insert/delete: иначе можно нарушить инварианты.
    """

    def __init__(self):
        self.root = None
        self._size = 0

    def __len__(self):
        return self._size

    def _height(self, node):
        return 0 if node is None else node.height

    def _update_height(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _balance(self, node):
        if node is None:
            return 0
        return self._height(node.left) - self._height(node.right)

    def _rotate_right(self, node):
        """Поднять левого ребёнка; сохранить порядок ключей поддеревьев."""
        new_root = node.left
        assert new_root is not None
        middle = new_root.right

        new_root.right = node
        node.left = middle

        # Нижний узел обновляем первым: его высота нужна новому корню.
        self._update_height(node)
        self._update_height(new_root)
        return new_root

    def _rotate_left(self, node):
        """Зеркальный вариант правого поворота."""
        new_root = node.right
        assert new_root is not None
        middle = new_root.left

        new_root.left = node
        node.right = middle

        self._update_height(node)
        self._update_height(new_root)
        return new_root

    def _rebalance(self, node):
        """Пересчитать высоту и при необходимости вернуть новый корень."""
        self._update_height(node)
        balance = self._balance(node)

        if balance > 1:
            # LR: сначала превращаем левый зигзаг в случай LL.
            if self._balance(node.left) < 0:
                assert node.left is not None
                node.left = self._rotate_left(node.left)
            # При балансе левого ребёнка 0 тоже нужен один правый поворот.
            return self._rotate_right(node)

        if balance < -1:
            # RL: сначала превращаем правый зигзаг в случай RR.
            if self._balance(node.right) > 0:
                assert node.right is not None
                node.right = self._rotate_right(node.right)
            # Нулевой баланс правого ребёнка также допускает один поворот.
            return self._rotate_left(node)

        return node

    def insert(self, value):
        """Добавить ключ. Вернуть False, если он уже присутствует."""
        self.root, inserted = self._insert(self.root, value)
        if inserted:
            self._size += 1
        return inserted

    def _insert(self, node, value):
        if node is None:
            return Node(value), True

        if value < node.value:
            node.left, inserted = self._insert(node.left, value)
        elif value > node.value:
            node.right, inserted = self._insert(node.right, value)
        else:
            return node, False

        # При возврате рекурсии проверяем каждого предка нового узла.
        if inserted:
            node = self._rebalance(node)
        return node, inserted

    def search(self, value):
        """Итеративный поиск без дополнительного рекурсивного стека."""
        node = self.root
        while node is not None:
            if value == node.value:
                return True
            if value < node.value:
                node = node.left
            else:
                node = node.right
        return False

    def delete(self, value):
        """Удалить ключ. Вернуть False, если такого ключа нет."""
        self.root, deleted = self._delete(self.root, value)
        # Даже при удалении узла с двумя детьми размер уменьшается один раз.
        if deleted:
            self._size -= 1
        return deleted

    def _delete(self, node, value):
        if node is None:
            return None, False

        if value < node.value:
            node.left, deleted = self._delete(node.left, value)
        elif value > node.value:
            node.right, deleted = self._delete(node.right, value)
        else:
            # Нет детей или только один: заменить узел оставшимся ребёнком.
            if node.left is None:
                return node.right, True
            if node.right is None:
                return node.left, True

            # Два ребёнка: взять следующий по величине ключ (successor).
            successor = node.right
            while successor.left is not None:
                successor = successor.left
            node.value = successor.value
            node.right, _ = self._delete(node.right, successor.value)
            deleted = True

        # После удаления уменьшение высоты может затронуть нескольких предков.
        if deleted:
            node = self._rebalance(node)
        return node, deleted

    def inorder(self):
        """Вернуть ключи по возрастанию: левое поддерево, узел, правое."""
        result = []

        def visit(node):
            if node is None:
                return
            visit(node.left)
            result.append(node.value)
            visit(node.right)

        visit(self.root)
        return result

    def to_ascii(self):
        """Показать корень, детей L/R, высоту h и баланс каждого узла.

        Пропущенный ребёнок показывается явно, если у узла есть второй ребёнок.
        У листьев пустые дети не печатаются.
        """
        if self.root is None:
            return "<empty AVL tree>"

        lines = []

        def draw(node, depth, label):
            prefix = "    " * depth
            if node is None:
                lines.append(f"{prefix}{label}: (empty)")
                return
            lines.append(
                f"{prefix}{label}: {node.value} "
                f"[h={node.height}, balance={self._balance(node)}]"
            )
            if node.left is not None or node.right is not None:
                draw(node.left, depth + 1, "L")
                draw(node.right, depth + 1, "R")

        draw(self.root, 0, "ROOT")
        return "\n".join(lines)

## Пример использования

`insert` возвращает `True`, если добавлен новый ключ;
`delete` — если ключ удалён; `search` — если ключ найден.

На рисунке `ROOT` — корень, `L` — левый ребёнок, `R` — правый,
`h` — высота, `balance` — разность высот.

In [3]:
tree = AVL()

for value in [4, 2, 5, 1, 3]:
    tree.insert(value)

print(tree.to_ascii())
print('Поиск 3:', tree.search(3))
print('Поиск 9:', tree.search(9))
print('Повторная вставка 3:', tree.insert(3))
print('Ключи по возрастанию:', tree.inorder())

ROOT: 4 [h=3, balance=1]
    L: 2 [h=2, balance=0]
        L: 1 [h=1, balance=0]
        R: 3 [h=1, balance=0]
    R: 5 [h=1, balance=0]
Поиск 3: True
Поиск 9: False
Повторная вставка 3: False
Ключи по возрастанию: [1, 2, 3, 4, 5]


## Визуализация поворотов

### LL: вставка 30, 20, 10

У 30 баланс стал +2. Поднимаем левого ребёнка правым поворотом.

```text
      30                 20
     /                  /  \
    20        →        10  30
   /
  10
```

### RR: вставка 10, 20, 30

У 10 баланс стал −2. Поднимаем правого ребёнка левым поворотом.

```text
  10                     20
    \                   /  \
     20       →        10  30
       \
        30
```

### LR: вставка 30, 10, 20

Сначала левый поворот у 10, затем правый у 30.

```text
     30             30                20
    /              /                 /  \
   10      →      20       →        10  30
     \           /
      20        10
```

### RL: вставка 10, 30, 20

Сначала правый поворот у 30, затем левый у 10.

```text
  10             10                   20
    \              \                 /  \
     30    →        20     →        10  30
    /                 \
   20                  30
```

Во всех четырёх случаях результат одинаковый: корень 20, дети 10 и 30.
Ниже печатается состояние после каждой вставки.

In [4]:
cases = {
    'LL': [30, 20, 10],
    'RR': [10, 20, 30],
    'LR': [30, 10, 20],
    'RL': [10, 30, 20],
}

for name, values in cases.items():
    example = AVL()
    print(f'\n{name}')

    for value in values:
        example.insert(value)
        print(f'После вставки {value}:')
        print(example.to_ascii())


LL
После вставки 30:
ROOT: 30 [h=1, balance=0]
После вставки 20:
ROOT: 30 [h=2, balance=1]
    L: 20 [h=1, balance=0]
    R: (empty)
После вставки 10:
ROOT: 20 [h=2, balance=0]
    L: 10 [h=1, balance=0]
    R: 30 [h=1, balance=0]

RR
После вставки 10:
ROOT: 10 [h=1, balance=0]
После вставки 20:
ROOT: 10 [h=2, balance=-1]
    L: (empty)
    R: 20 [h=1, balance=0]
После вставки 30:
ROOT: 20 [h=2, balance=0]
    L: 10 [h=1, balance=0]
    R: 30 [h=1, balance=0]

LR
После вставки 30:
ROOT: 30 [h=1, balance=0]
После вставки 10:
ROOT: 30 [h=2, balance=1]
    L: 10 [h=1, balance=0]
    R: (empty)
После вставки 20:
ROOT: 20 [h=2, balance=0]
    L: 10 [h=1, balance=0]
    R: 30 [h=1, balance=0]

RL
После вставки 10:
ROOT: 10 [h=1, balance=0]
После вставки 30:
ROOT: 10 [h=2, balance=-1]
    L: (empty)
    R: 30 [h=1, balance=0]
После вставки 20:
ROOT: 20 [h=2, balance=0]
    L: 10 [h=1, balance=0]
    R: 30 [h=1, balance=0]


## Визуализация удаления

Продолжим пример с ключами `[4, 2, 5, 1, 3]`.

1. Удаляем лист 5.
2. У 4 слева высота 2, справа 0: баланс стал +2.
3. У левого ребёнка 2 баланс равен 0. Достаточно одного правого поворота.

```text
До удаления:      После удаления 5:     После поворота:
      4                  4                    2
     / \                /                    / \
    2   5              2                    1   4
   / \                / \                      /
  1   3              1   3                    3
```

Теперь удалим корень 2 с двумя детьми. Минимум справа — 3:
он заменяет 2, а прежний узел с 3 удаляется.

```text
      2                  3
     / \                / \
    1   4      →       1   4
       /
      3
```

In [5]:
print('Удаляем 5:')
tree.delete(5)
print(tree.to_ascii())

print('\nУдаляем 2:')
tree.delete(2)
print(tree.to_ascii())
print('Ключи:', tree.inorder())
print('Размер:', len(tree))

Удаляем 5:
ROOT: 2 [h=3, balance=-1]
    L: 1 [h=1, balance=0]
    R: 4 [h=2, balance=1]
        L: 3 [h=1, balance=0]
        R: (empty)

Удаляем 2:
ROOT: 3 [h=2, balance=0]
    L: 1 [h=1, balance=0]
    R: 4 [h=1, balance=0]
Ключи: [1, 3, 4]
Размер: 3


## Оценка сложности

Пусть `n` — число узлов.

AVL поддерживает высоту `O(log n)`. Поиск, вставка и удаление проходят
по пути в дереве; один поворот занимает `O(1)`.

| Операция | Время | Дополнительная память |
|---|---|---|
| Поиск | `O(log n)` | `O(1)` — используется цикл |
| Вставка | `O(log n)` | `O(log n)` — стек рекурсии |
| Удаление | `O(log n)` | `O(log n)` — стек рекурсии |
| Обход `inorder` | `O(n)` | `O(n)` для списка и `O(log n)` для стека |

Оценки для основных операций даны для худшего случая.
Сравнение целых ключей считаем операцией `O(1)`.
Само дерево занимает `O(n)` памяти.

Для картинки из `L` символов `to_ascii` требует `O(n + L)` времени
и `O(L)` памяти для текста, плюс `O(log n)` для стека.
Стоимость печати не входит в сложность вставки и удаления.

## Тесты

Проверяем пустое дерево, дубликаты, поиск, удаление узлов с разным числом
детей и четыре вида поворотов. После изменений независимо пересчитываем
высоты и проверяем порядок ключей, баланс и размер.

Дополнительно перебираем 576 сочетаний порядка вставки и удаления
четырёх ключей и сравниваем 3000 случайных операций с множеством `set`.

In [6]:
import itertools
import random
import unittest



class TestAVL(unittest.TestCase):
    def assert_valid(self, tree, expected):
        """Независимо пересчитать высоты, порядок, баланс и число узлов."""
        visited = set()

        def inspect(node, lower=None, upper=None):
            if node is None:
                return 0, 0
            self.assertNotIn(id(node), visited, "Цикл или общий узел в дереве")
            visited.add(id(node))
            if lower is not None:
                self.assertGreater(node.value, lower)
            if upper is not None:
                self.assertLess(node.value, upper)
            left_height, left_count = inspect(node.left, lower, node.value)
            right_height, right_count = inspect(node.right, node.value, upper)
            actual_height = 1 + max(left_height, right_height)
            self.assertEqual(node.height, actual_height)
            self.assertLessEqual(abs(left_height - right_height), 1)
            return actual_height, 1 + left_count + right_count

        _, count = inspect(tree.root)
        self.assertEqual(count, len(expected))
        self.assertEqual(len(tree), len(expected))
        self.assertEqual(tree.inorder(), sorted(expected))

    def build(self, values):
        tree = AVL()
        expected = set()
        self.assert_valid(tree, expected)
        for value in values:
            self.assertEqual(tree.insert(value), value not in expected)
            expected.add(value)
            self.assert_valid(tree, expected)
        return tree, expected

    def remove(self, tree, expected, value):
        self.assertEqual(tree.delete(value), value in expected)
        expected.discard(value)
        self.assert_valid(tree, expected)

    def test_empty_tree(self):
        tree = AVL()
        self.assertFalse(tree.search(10))
        self.assertFalse(tree.delete(10))
        self.assertEqual(tree.inorder(), [])
        self.assertEqual(tree.to_ascii(), "<empty AVL tree>")
        self.assert_valid(tree, set())

    def test_node_defaults(self):
        node = Node(7)
        self.assertEqual(node.value, 7)
        self.assertIsNone(node.left)
        self.assertIsNone(node.right)
        self.assertEqual(node.height, 1)

    def test_insert_rotations(self):
        cases = {
            "LL": [30, 20, 10],
            "RR": [10, 20, 30],
            "LR": [30, 10, 20],
            "RL": [10, 30, 20],
        }
        for name, values in cases.items():
            with self.subTest(rotation=name):
                tree, _ = self.build(values)
                self.assertEqual(tree.root.value, 20)
                self.assertEqual(tree.root.left.value, 10)
                self.assertEqual(tree.root.right.value, 30)

    def test_duplicate_insert(self):
        tree, expected = self.build([20, 10, 30])
        before = tree.to_ascii()
        self.assertFalse(tree.insert(20))
        self.assert_valid(tree, expected)
        self.assertEqual(tree.to_ascii(), before)

    def test_search_existing_and_absent(self):
        tree, expected = self.build([0, -10, 20, -30, 5, 100])
        for value in expected:
            self.assertTrue(tree.search(value))
        for value in [-100, -1, 1, 99, 101]:
            self.assertFalse(tree.search(value))
        self.assert_valid(tree, expected)

    def test_delete_absent_does_not_change_tree(self):
        tree, expected = self.build([20, 10, 30])
        before = tree.to_ascii()
        self.remove(tree, expected, 99)
        self.assertEqual(tree.to_ascii(), before)

    def test_delete_only_root_and_reinsert(self):
        tree, expected = self.build([7])
        self.remove(tree, expected, 7)
        self.assertIsNone(tree.root)
        self.remove(tree, expected, 7)
        self.assertTrue(tree.insert(8))
        expected.add(8)
        self.assert_valid(tree, expected)

    def test_delete_leaf(self):
        tree, expected = self.build([20, 10, 30])
        self.remove(tree, expected, 10)
        self.assertIsNone(tree.root.left)

    def test_delete_root_with_one_child(self):
        for values, new_root in [([20, 10], 10), ([20, 30], 30)]:
            with self.subTest(values=values):
                tree, expected = self.build(values)
                self.remove(tree, expected, 20)
                self.assertEqual(tree.root.value, new_root)

    def test_delete_non_root_with_one_child(self):
        tree, expected = self.build([20, 10, 30, 5])
        self.remove(tree, expected, 10)
        self.assertEqual(tree.root.left.value, 5)

    def test_delete_two_children_deep_successor(self):
        tree, expected = self.build([40, 20, 60, 10, 30, 50, 70, 45, 55])
        self.remove(tree, expected, 40)
        self.assertEqual(tree.root.value, 45)
        self.assertEqual(len(tree), 8)
        self.assertFalse(tree.search(40))

    def test_delete_two_children_successor_with_right_child(self):
        tree, expected = self.build([40, 20, 60, 10, 30, 50, 70, 55])
        self.remove(tree, expected, 40)
        self.assertEqual(tree.root.value, 50)
        self.assertTrue(tree.search(55))
        self.assertEqual(len(tree), 7)

    def test_delete_rotations(self):
        cases = [
            ("LL", [5, 2, 6, 1], 6, 2),
            ("RR", [2, 1, 5, 6], 1, 5),
            ("LR", [5, 2, 6, 4], 6, 4),
            ("RL", [2, 1, 5, 3], 1, 3),
            ("left child balance 0", [4, 2, 5, 1, 3], 5, 2),
            ("right child balance 0", [2, 1, 4, 3, 5], 1, 4),
        ]
        for name, values, removed, root in cases:
            with self.subTest(rotation=name):
                tree, expected = self.build(values)
                self.remove(tree, expected, removed)
                self.assertEqual(tree.root.value, root)

    def test_sorted_insertions_and_full_deletion(self):
        for values in [list(range(100)), list(reversed(range(100)))]:
            with self.subTest(first=values[0]):
                tree, expected = self.build(values)
                for value in range(0, 100, 2):
                    self.remove(tree, expected, value)
                for value in range(99, 0, -2):
                    self.remove(tree, expected, value)
                self.assertIsNone(tree.root)

    def test_all_small_insert_and_delete_orders(self):
        # Все 24 порядка вставки и 24 порядка удаления четырёх разных ключей.
        orders = list(itertools.permutations([0, 1, 2, 3]))
        for inserted in orders:
            for removed in orders:
                with self.subTest(inserted=inserted, removed=removed):
                    tree, expected = self.build(inserted)
                    for value in removed:
                        self.remove(tree, expected, value)

    def test_randomized_against_set(self):
        for seed in [980, 17, 20261102]:
            with self.subTest(seed=seed):
                rng = random.Random(seed)
                tree = AVL()
                expected = set()
                for _ in range(1000):
                    value = rng.randrange(-50, 51)
                    operation = rng.choice(["insert", "delete", "search"])
                    if operation == "insert":
                        self.assertEqual(tree.insert(value), value not in expected)
                        expected.add(value)
                    elif operation == "delete":
                        self.assertEqual(tree.delete(value), value in expected)
                        expected.discard(value)
                    else:
                        self.assertEqual(tree.search(value), value in expected)
                    self.assert_valid(tree, expected)

    def test_ascii_labels_heights_and_missing_child(self):
        tree, _ = self.build([20, 10])
        self.assertEqual(
            tree.to_ascii(),
            "ROOT: 20 [h=2, balance=1]\n"
            "    L: 10 [h=1, balance=0]\n"
            "    R: (empty)",
        )


import sys

suite = unittest.TestLoader().loadTestsFromTestCase(TestAVL)
runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful()

test_all_small_insert_and_delete_orders (__main__.TestAVL.test_all_small_insert_and_delete_orders) ... 

ok


test_ascii_labels_heights_and_missing_child (__main__.TestAVL.test_ascii_labels_heights_and_missing_child) ... 

ok


test_delete_absent_does_not_change_tree (__main__.TestAVL.test_delete_absent_does_not_change_tree) ... 

ok


... (25 outputs left out)

ok


test_sorted_insertions_and_full_deletion (__main__.TestAVL.test_sorted_insertions_and_full_deletion) ... 

ok


----------------------------------------------------------------------
Ran 17 tests in 0.121s

OK
